# 00 · Data Audit — Component 3 (Corneal Cellular Intelligence Platform)

**Phase 0, Step 0.5.** Read-only analysis of the NEBSL specular-microscopy dataset and its
Konan CellChek D ground truth. Nothing here changes data or splits.

| # | Section | Output (in `outputs/audit/`) |
|---|---------|------------------------------|
| 1 | Dataset counts & exclusions | `01_counts.csv` |
| 2 | Ground-truth distributions | `02_gt_distributions.png`, `02_gt_summary.csv` |
| 3 | Biomarker relationships | `03_relationships.png` |
| 4 | Grade / eye / year mix | `04_metadata_mix.png` |
| 5 | Split balance | `05_split_balance.png`, `05_split_balance.csv` |
| 6 | Image quality metrics | `06_image_quality.png`, `06_image_quality.csv` |
| 7 | Stress lines / dark bands (heuristic + visual check) | `07_*.png` |
| 8 | Example gallery (best / typical / worst) | `08_gallery.png` |
| 9 | Cell size from FFT | `09_cell_spacing.png` |
| 10 | Audit summary | `audit_summary.json` |

**Run on Colab:** Runtime → Run all. A CPU runtime is enough (no GPU needed).

## 0 · Setup

In [ ]:
# Works on Colab (mounts Drive, pulls the repo, copies images to fast disk) and locally.
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RESEARCH = "/content/drive/MyDrive/Colab Notebooks/Research_Project"
    REPO_DIR = f"{RESEARCH}/xai-cornea-grading"
    LOCAL_DATA = "/content/data"

    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

    # Copy images once per session — reading from local disk is much faster than Drive.
    if not Path(LOCAL_DATA, "Finalized_images").exists():
        os.makedirs(LOCAL_DATA, exist_ok=True)
        subprocess.run(["cp", "-r", f"{RESEARCH}/Finalized_images", LOCAL_DATA], check=True)
        subprocess.run(["cp", f"{RESEARCH}/validation_ground_truth_dataset.xlsx", LOCAL_DATA], check=True)

    os.environ["CELLULAR_DATA_ROOT"] = LOCAL_DATA
    os.environ["CELLULAR_OUTPUT_ROOT"] = f"{RESEARCH}/cellular_outputs"
    os.environ["CELLULAR_MODELS_ROOT"] = f"{RESEARCH}/cellular_models"
    COMPONENT_DIR = Path(REPO_DIR, "ml", "cellular")
else:
    COMPONENT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(COMPONENT_DIR)
if str(COMPONENT_DIR) not in sys.path:
    sys.path.insert(0, str(COMPONENT_DIR))
print("Component dir:", COMPONENT_DIR)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from scipy import stats
from tqdm.auto import tqdm

from config import load_config

cfg = load_config()
P = cfg["paths"]
OUT = P["outputs_dir"] / "audit"
OUT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
SPLIT_ORDER = ["train", "calibration", "val", "test"]
SEED = cfg["split"]["seed"]


def save(fig, name):
    """Save a figure to outputs/audit/ and show it."""
    fig.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def load_gray(image_file):
    """Read one specular image as a uint8 grayscale array."""
    return cv2.imread(str(P["images_dir"] / image_file), cv2.IMREAD_GRAYSCALE)


print("Images :", P["images_dir"])
print("Outputs:", OUT)

## 1 · Dataset counts & exclusions
Every image and ground-truth row is accounted for; exclusions carry a reason (from `splits/manifest.csv`).

In [ ]:
man = pd.read_csv(P["splits_dir"] / "manifest.csv")
split_summary = json.loads((P["splits_dir"] / "split_summary.json").read_text())
for col in ("donor_age", "year", "ecd", "cv", "hex", "mean_area", "num_cells"):
    man[col] = pd.to_numeric(man[col], errors="coerce")

inc = man[man["status"] == "included"].copy()

counts = pd.Series({
    "image files": int((man["status"] != "excluded:missing_image").sum()),
    "ground-truth rows": int(man["cornea_id"].notna().sum()),
    "included": len(inc),
    "needs_review (held out)": int((man["status"] == "review:needs_review").sum()),
    "excluded": int(man["status"].str.startswith("excluded").sum()),
    "unique tissues (included)": int(inc["group_id"].nunique()),
    "tissues with >1 capture": int((inc.groupby("group_id").size() > 1).sum()),
}, name="count")
counts.to_csv(OUT / "01_counts.csv")

display(counts.to_frame())
display(man["status"].value_counts().to_frame("images"))
display(man.loc[man["status"] != "included", ["image_file", "status"]])

## 2 · Ground-truth distributions
Konan CellChek D values for included images. Red dashed lines: clinical thresholds from the scope
(ECD < 2000 = poor viability; HEX < 60% = below healthy).

In [ ]:
BIOMARKERS = {"ecd": "ECD (cells/mm²)", "cv": "CV (%)", "hex": "HEX (%)",
              "mean_area": "Mean cell area (µm²)", "num_cells": "Cells analysed by Konan"}
REF_LINES = {"ecd": [2000], "hex": [60]}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (col, label) in zip(axes, BIOMARKERS.items()):
    sns.histplot(inc[col], bins=40, ax=ax, color="tab:blue")
    for x in REF_LINES.get(col, []):
        ax.axvline(x, color="tab:red", ls="--", lw=1)
    ax.set_xlabel(label)
    ax.set_ylabel("images")
fig.suptitle("Konan ground-truth distributions (included images)")
save(fig, "02_gt_distributions")

gt_desc = inc[list(BIOMARKERS)].describe(percentiles=[.05, .25, .5, .75, .95]).T.round(1)
gt_desc.to_csv(OUT / "02_gt_summary.csv")
display(gt_desc)
print(f"ECD < 2000 (poor viability): {(inc['ecd'] < 2000).mean():.1%}")
print(f"HEX < 60%  (below healthy):  {(inc['hex'] < 60).mean():.1%}")
print(f"Konan cells analysed per image: median {inc['num_cells'].median():.0f}")

## 3 · Biomarker relationships
* **ECD vs mean area** should follow ECD = 10⁶ / mean area (Konan computes ECD from the analysed cell area).
  This matters for Phase 4–5: ECD accuracy depends directly on cell-area accuracy and the µm/px scale.
* **CV vs HEX** — polymegethism and pleomorphism usually move together.
* **ECD vs donor age** — ECD is expected to decline with age.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(inc["mean_area"], inc["ecd"], s=6, alpha=.5)
xs = np.linspace(inc["mean_area"].min(), inc["mean_area"].max(), 200)
ax.plot(xs, 1e6 / xs, "r-", lw=1.5, label="ECD = 10⁶ / mean area")
ax.set(xlabel="Mean cell area (µm²)", ylabel="ECD (cells/mm²)", title="ECD vs mean area")
ax.legend()

ax = axes[1]
r_cv_hex, _ = stats.pearsonr(inc["cv"], inc["hex"])
ax.scatter(inc["cv"], inc["hex"], s=6, alpha=.5)
ax.set(xlabel="CV (%)", ylabel="HEX (%)", title=f"CV vs HEX  (r = {r_cv_hex:.2f})")

ax = axes[2]
d = inc.dropna(subset=["donor_age"])
age_fit = stats.linregress(d["donor_age"], d["ecd"])
ages = np.linspace(d["donor_age"].min(), d["donor_age"].max(), 100)
ax.scatter(d["donor_age"], d["ecd"], s=6, alpha=.5)
ax.plot(ages, age_fit.intercept + age_fit.slope * ages, "r-")
ax.set(xlabel="Donor age (years)", ylabel="ECD (cells/mm²)",
       title=f"ECD vs age  (slope {age_fit.slope:.1f}/yr, r = {age_fit.rvalue:.2f})")
save(fig, "03_relationships")

ecd_area_diff = ((inc["ecd"] - 1e6 / inc["mean_area"]).abs() / inc["ecd"])
print(f"|ECD − 10⁶/mean_area| / ECD : median {ecd_area_diff.median():.2%}, "
      f"95th pct {ecd_area_diff.quantile(.95):.2%}, max {ecd_area_diff.max():.2%}")

## 4 · Grade, eye and year mix
Konan reliability grade (green / yellow / red), eye (OD = right, OS = left) and acquisition year.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["grade", "eye", "year"]):
    vc = inc[col].astype("string").fillna("missing").value_counts().sort_index()
    ax.bar(vc.index, vc.values)
    ax.set(title=col, ylabel="images")
    for i, v in enumerate(vc.values):
        ax.text(i, v, str(v), ha="center", va="bottom")
save(fig, "04_metadata_mix")

## 5 · Split balance
The frozen tissue-level split should have similar biomarker distributions in every split.
KS test p-values > 0.05 mean no detectable difference between train and test.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ["ecd", "cv", "hex"]):
    sns.kdeplot(data=inc, x=col, hue="split", hue_order=SPLIT_ORDER, common_norm=False, ax=ax)
    ax.set_xlabel(BIOMARKERS[col])
fig.suptitle("Biomarker distributions per split")
save(fig, "05_split_balance")

balance = (inc.groupby("split")[["ecd", "cv", "hex", "mean_area"]]
              .agg(["mean", "std"]).round(1).reindex(SPLIT_ORDER))
balance.insert(0, ("n", "images"), inc["split"].value_counts().reindex(SPLIT_ORDER))
balance.to_csv(OUT / "05_split_balance.csv")
display(balance)

ks_pvalues = {}
for col in ["ecd", "cv", "hex"]:
    ks = stats.ks_2samp(inc.loc[inc["split"] == "train", col], inc.loc[inc["split"] == "test", col])
    ks_pvalues[col] = round(float(ks.pvalue), 4)
print("KS train vs test p-values:", ks_pvalues)

## 6 · Image quality metrics
Computed for every included image (about 1 minute):

| Metric | Meaning |
|--------|---------|
| `brightness`, `contrast` | mean and std of pixel intensity |
| `sharpness` | variance of the Laplacian — low = blurred |
| `illum_unevenness` | std / mean of the smooth illumination field — high = uneven lighting |
| `dark_fraction` | share of the image whose illumination is very dark (< 40) |
| `textured_fraction` | share of the image with clear cell-scale texture — a proxy for analysable area |
| `dark_band_score` | depth of the strongest dark band (stress line / fold), see Section 7 |

`quality_score` (0–100) ranks images by sharpness + textured area − unevenness − darkness.
It is a descriptive audit measure only; the real ROI quality model is built in Phase 6.

In [ ]:
def dark_band_score(background):
    """
    Stress lines and folds in these images look like long dark bands.
    Score = deepest dip of the row / column intensity profile below its median,
    ignoring a 10% margin (image borders are naturally darker).
    """
    norm = background / (np.median(background) + 1e-6)
    scores = []
    for axis in (0, 1):
        prof = norm.mean(axis=axis).astype(np.float32)
        prof = cv2.GaussianBlur(prof.reshape(1, -1), (0, 0), 2).ravel()
        n = len(prof)
        core = prof[int(.1 * n): int(.9 * n)]
        scores.append(1.0 - core.min() / (np.median(prof) + 1e-6))
    return float(max(scores))


def quality_metrics(img):
    f = img.astype(np.float32)
    small = cv2.resize(f, None, fx=0.25, fy=0.25, interpolation=cv2.INTER_AREA)
    background = cv2.GaussianBlur(small, (0, 0), 12)            # illumination field
    texture = f - cv2.GaussianBlur(f, (0, 0), 6)                 # cell-scale detail
    tex_energy = np.sqrt(cv2.GaussianBlur(texture ** 2, (0, 0), 16))
    return {
        "brightness": float(f.mean()),
        "contrast": float(f.std()),
        "sharpness": float(cv2.Laplacian(f, cv2.CV_32F).var()),
        "illum_unevenness": float(background.std() / (background.mean() + 1e-6)),
        "dark_fraction": float((background < 40).mean()),
        "textured_fraction": float((tex_energy > 0.5 * np.percentile(tex_energy, 95)).mean()),
        "dark_band_score": dark_band_score(background),
    }


rows = []
for _, r in tqdm(inc.iterrows(), total=len(inc), desc="Quality metrics"):
    rows.append({"image_id": r["image_id"], **quality_metrics(load_gray(r["image_file"]))})
quality = pd.DataFrame(rows)

def z(s):
    return (s - s.mean()) / (s.std() + 1e-9)

raw_score = (z(np.log1p(quality["sharpness"])) + z(quality["textured_fraction"])
             - z(quality["illum_unevenness"]) - z(quality["dark_fraction"]))
quality["quality_score"] = (raw_score.rank(pct=True) * 100).round(1)

inc = inc.merge(quality, on="image_id", how="left")
quality.to_csv(OUT / "06_image_quality.csv", index=False)

QUALITY_COLS = ["brightness", "contrast", "sharpness", "illum_unevenness",
                "dark_fraction", "textured_fraction"]
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, col in zip(axes.ravel(), QUALITY_COLS):
    sns.histplot(inc[col], bins=40, ax=ax)
fig.suptitle("Image quality metrics (included images)")
save(fig, "06_image_quality")

display(inc[QUALITY_COLS + ["dark_band_score", "quality_score"]].describe().T.round(3))

# Does image quality relate to the Konan measurements?
for col in ["ecd", "num_cells", "hex"]:
    rho, p = stats.spearmanr(inc["quality_score"], inc[col])
    print(f"Spearman quality_score vs {col:9s}: rho = {rho:+.2f} (p = {p:.3f})")

## 7 · Stress lines / dark bands — heuristic estimate + visual check
An image is **flagged** when `dark_band_score` exceeds `DARK_BAND_THRESHOLD`. This is a heuristic,
so it must be checked by eye:

1. Look at the two galleries below (flagged vs not flagged, random samples).
2. Count the mistakes and enter them in the next cell.
3. If agreement is poor, adjust `DARK_BAND_THRESHOLD` and re-run this section.

In [ ]:
DARK_BAND_THRESHOLD = 0.35   # tune after the visual check
N_SHOW = 12

inc["dark_band_flag"] = inc["dark_band_score"] > DARK_BAND_THRESHOLD
print(f"Flagged as having a dark band / stress line: "
      f"{inc['dark_band_flag'].sum()} of {len(inc)} ({inc['dark_band_flag'].mean():.1%})")

def gallery(df, title, name, ncols=6):
    nrows = int(np.ceil(len(df) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), df.iterrows()):
        img = load_gray(r["image_file"])
        ax.imshow(cv2.resize(img, None, fx=0.25, fy=0.25, interpolation=cv2.INTER_AREA), cmap="gray")
        ax.set_title(f"{r['image_id'][:22]}\nscore {r['dark_band_score']:.2f}", fontsize=8)
    fig.suptitle(title)
    save(fig, name)

flagged_sample = inc[inc["dark_band_flag"]].sample(min(N_SHOW, inc["dark_band_flag"].sum()), random_state=SEED)
clean_sample = inc[~inc["dark_band_flag"]].sample(N_SHOW, random_state=SEED)
gallery(flagged_sample, "FLAGGED — should show dark bands / stress lines", "07_flagged_sample")
gallery(clean_sample, "NOT flagged — should be free of dark bands", "07_clean_sample")

In [ ]:
# Enter your visual check here (count from the two galleries above).
WRONGLY_FLAGGED = None   # flagged images that actually have NO dark band
MISSED = None            # not-flagged images that actually DO have a dark band

if WRONGLY_FLAGGED is None or MISSED is None:
    print("Fill in WRONGLY_FLAGGED and MISSED after looking at the galleries, then re-run this cell.")
    visual_agreement = None
else:
    n_checked = len(flagged_sample) + len(clean_sample)
    visual_agreement = 1 - (WRONGLY_FLAGGED + MISSED) / n_checked
    print(f"Heuristic agreement with visual check: {visual_agreement:.0%} "
          f"({n_checked} images checked)")

## 8 · Example gallery — best, typical and worst images (by `quality_score`)

In [ ]:
ranked = inc.sort_values("quality_score")
picks = pd.concat([
    ranked.tail(3).assign(group="best"),
    ranked.iloc[len(ranked) // 2 - 1: len(ranked) // 2 + 2].assign(group="typical"),
    ranked.head(3).assign(group="worst"),
])
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
    ax.imshow(load_gray(r["image_file"]), cmap="gray")
    ax.set_title(f"{r['group'].upper()} · q={r['quality_score']:.0f} · ECD {r['ecd']:.0f} · "
                 f"HEX {r['hex']:.0f}\n{r['image_id']}", fontsize=9)
    ax.axis("off")
fig.suptitle("Image quality examples")
save(fig, "08_gallery")

## 9 · Cell size from the FFT
Endothelial cells form a roughly hexagonal lattice, which gives a ring in the 2-D power spectrum.
The ring radius gives the lattice row spacing; for a hexagonal lattice the cell centre-to-centre
distance is `d = 2 · spacing / √3`.

This confirms the cell size in **pixels**, which decides Phase 2 crop sizes. On the **calibration
split only** it also gives a *preliminary* µm/px estimate (hexagon area A = (√3/2)·d², using Konan's
mean area) — the real calibration is Phase 4.

In [ ]:
CROP = 512
FFT_SAMPLE = 200


def radial_power(img, crop=CROP):
    """Radially averaged, 1/f-compensated power spectrum of the most textured crop."""
    f = img.astype(np.float32)
    f = f - cv2.GaussianBlur(f, (0, 0), 10)                        # remove illumination
    energy = cv2.GaussianBlur(f ** 2, (0, 0), 40)
    h, w = f.shape
    energy[: crop // 2, :] = energy[h - crop // 2:, :] = 0          # keep crop inside image
    energy[:, : crop // 2] = energy[:, w - crop // 2:] = 0
    cy, cx = np.unravel_index(np.argmax(energy), energy.shape)
    c = f[cy - crop // 2: cy + crop // 2, cx - crop // 2: cx + crop // 2]
    c = c * np.outer(np.hanning(crop), np.hanning(crop))
    power = np.abs(np.fft.fftshift(np.fft.fft2(c))) ** 2
    yy, xx = np.indices(power.shape)
    r = np.hypot(yy - crop // 2, xx - crop // 2).astype(int)
    radial = np.bincount(r.ravel(), power.ravel()) / np.maximum(np.bincount(r.ravel()), 1)
    return radial * np.arange(len(radial))                          # compensate 1/f fall-off


def cell_spacing_px(img, crop=CROP, min_px=8, max_px=60):
    radial = radial_power(img, crop)
    k_lo, k_hi = int(crop / max_px), int(crop / min_px)
    k = k_lo + int(np.argmax(radial[k_lo:k_hi]))
    row_spacing = crop / k
    return 2 * row_spacing / np.sqrt(3)


sample = inc.sample(min(FFT_SAMPLE, len(inc)), random_state=SEED)
sample = pd.concat([sample, inc[inc["split"] == "calibration"]]).drop_duplicates("image_id")
sample["cell_spacing_px"] = [cell_spacing_px(load_gray(f))
                             for f in tqdm(sample["image_file"], desc="FFT")]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
example = load_gray(sample.iloc[0]["image_file"])
rp = radial_power(example)
axes[0].plot(np.arange(len(rp))[5:CROP // 4], rp[5:CROP // 4])
axes[0].set(xlabel="spatial frequency (cycles / crop)", ylabel="power × f",
            title=f"Radial spectrum, {sample.iloc[0]['image_id']}")
sns.histplot(sample["cell_spacing_px"], bins=30, ax=axes[1])
axes[1].set(xlabel="cell centre-to-centre distance (px)", title="Estimated cell size")
save(fig, "09_cell_spacing")

spacing_med = float(sample["cell_spacing_px"].median())
print(f"Median cell centre-to-centre distance: {spacing_med:.1f} px "
      f"(IQR {sample['cell_spacing_px'].quantile(.25):.1f}–{sample['cell_spacing_px'].quantile(.75):.1f})")
print(f"A 256×256 crop holds roughly {(256 / spacing_med) ** 2 * 2 / np.sqrt(3):.0f} cells")

cal = sample[sample["split"] == "calibration"].copy()
cal["d_um"] = np.sqrt(2 * cal["mean_area"] / np.sqrt(3))
cal["um_per_px_est"] = cal["d_um"] / cal["cell_spacing_px"]
um_per_px_prelim = float(cal["um_per_px_est"].median())
print(f"PRELIMINARY µm/px (calibration split, n={len(cal)}): {um_per_px_prelim:.3f} "
      f"(IQR {cal['um_per_px_est'].quantile(.25):.3f}–{cal['um_per_px_est'].quantile(.75):.3f}) "
      "— final value comes from Phase 4")

## 10 · Audit summary

In [ ]:
audit = {
    "counts": {k: int(v) for k, v in counts.items()},
    "ground_truth": {c: {"mean": round(float(inc[c].mean()), 2), "sd": round(float(inc[c].std()), 2),
                         "min": float(inc[c].min()), "max": float(inc[c].max())} for c in BIOMARKERS},
    "share_ecd_below_2000": round(float((inc["ecd"] < 2000).mean()), 4),
    "share_hex_below_60": round(float((inc["hex"] < 60).mean()), 4),
    "ecd_vs_inverse_mean_area_median_rel_diff": round(float(ecd_area_diff.median()), 4),
    "cv_hex_pearson_r": round(float(r_cv_hex), 3),
    "ecd_age_slope_per_year": round(float(age_fit.slope), 2),
    "grade_counts": {str(k): int(v) for k, v in inc["grade"].value_counts().items()},
    "ks_train_vs_test_pvalues": ks_pvalues,
    "dark_band_threshold": DARK_BAND_THRESHOLD,
    "dark_band_flagged_share": round(float(inc["dark_band_flag"].mean()), 4),
    "dark_band_visual_agreement": visual_agreement,
    "median_cell_spacing_px": round(spacing_med, 2),
    "preliminary_um_per_px_calibration_split": round(um_per_px_prelim, 4),
}
(OUT / "audit_summary.json").write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))
print("\nSaved to", OUT)

## Findings (fill in after running)
* **Dataset:** …
* **Ground truth:** …
* **Image quality / artifacts:** …
* **Cell size → Phase 2 crop size:** …
* **Limitations:** nearly all images are Konan grade *green*; eye-to-donor grouping unconfirmed.